# RAG Pipeline Practice Notebook

This notebook demonstrates a complete RAG pipeline including:
1. Non-RAG / Bruteforce approach
2. PDF Extraction with Docling
3. Chunking & Inspection
4. Embeddings (HuggingFace)
5. Vector Database (Chroma)
6. Retrieval & Generation (Groq)


In [ ]:
import os
import sys
from dotenv import load_dotenv

# Load environment variables (ensure .env exists with GROQ_API_KEY)
load_dotenv()

# Check key
if not os.getenv("GROQ_API_KEY"):
    print("WARNING: GROQ_API_KEY not found in environment variables.")
else:
    print("GROQ_API_KEY loaded successfully.")



## 1. Simple Function Pipeline / Bruteforce Approach

Before using valid RAG, let's see how a simple keyword search (bruteforce) works on raw text.


In [ ]:
def simple_bruteforce_search(text_corpus, query):
    """
    Searches for the query string directly in the text corpus.
    """
    print(f"Searching for: '{query}'")
    results = []
    
    # Split corpus into pseudo-sentences or lines for 'chunks'
    lines = text_corpus.split('\n')
    
    for i, line in enumerate(lines):
        if query.lower() in line.lower():
            results.append((i, line.strip()))
            
    return results

# Dummy Corpus
dummy_text = """
Paracetamol is a common pain reliever and fever reducer.
Ibuprofen is a nonsteroidal anti-inflammatory drug (NSAID).
Antibiotics are used to treat bacterial infections.
Vitamin C is essential for the repair of all body tissues.
"""

# Test
query = "fever"
matches = simple_bruteforce_search(dummy_text, query)

print(f"\nFound {len(matches)} matches:")
for idx, content in matches:
    print(f"Line {idx}: {content}")



## 2. PDF Data Extraction using Docling

We will now use `Docling` to extract text from a real PDF.


In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableStructureOptions, AcceleratorOptions, AcceleratorDevice
from docling.document_converter import PdfFormatOption
import warnings

warnings.filterwarnings("ignore")

# configure Docling options
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = True
pipeline_options.accelerator_options = AcceleratorOptions(num_threads=4, device=AcceleratorDevice.AUTO)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# Define PDF Path - Change this to a valid path on your system
# pdf_path = "data/Drug Interactions.pdf" 
# Or use an absolute path if needed. Assuming running from project root or notebook folder.
# If running from 'notebook/' folder, go up one level.
pdf_path = "../data/Drug Interactions.pdf" 

if not os.path.exists(pdf_path):
    print(f"File not found: {pdf_path}. Please update variable 'pdf_path' to a valid PDF file.")
else:
    print(f"Processing {pdf_path}...")
    conversion_result = converter.convert(pdf_path)
    doc = conversion_result.document
    
    # Export to markdown
    markdown_content = doc.export_to_markdown()
    print("Conversion Complete.")
    print(f"Extracted {len(markdown_content)} characters.")



## 3. Chunking & Inspection

Let's chunk the document and inspect how it looks.


In [ ]:
from docling.chunking import HybridChunker

# Use the same tokenizer as the embedding model roughly
chunker = HybridChunker(tokenizer="intfloat/multilingual-e5-large")

# Iterate over chunks
print("Chunking document...")
chunk_iter = chunker.chunk(dl_doc=doc)
chunks = list(chunk_iter)

print(f"Total Chunks generated: {len(chunks)}")

# Inspect the first few chunks
print("\n--- Chunk 1 Preview ---")
print(chunker.serialize(chunk=chunks[0]))

print("\n--- Chunk 2 Preview ---")
if len(chunks) > 1:
    print(chunker.serialize(chunk=chunks[1]))



## 4. Embeddings (HuggingFace)

Using `langchain_huggingface` to load an intermediate level embedding model.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_name = "intfloat/multilingual-e5-large"

print(f"Loading embedding model: {embedding_model_name}")
embedding_function = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': 'cpu'}, # Set to 'cuda' if you have a GPU
    encode_kwargs={'normalize_embeddings': True}
)

# Test embedding generation
test_emb = embedding_function.embed_query("This is a test sentence.")
print(f"Embedding vector length: {len(test_emb)}")



## 5 & 6. Vector Store (Chroma) & Retrieval

Storing chunks in ChromaDB and retrieving relevant ones.


In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document as LCDocument

# 1. Convert Docling chunks to LangChain Documents
lc_docs = []
for i, chunk in enumerate(chunks):
    text = chunker.serialize(chunk=chunk)
    metadata = {
        "source": pdf_path,
        "chunk_index": i
    }
    lc_docs.append(LCDocument(page_content=text, metadata=metadata))

print(f"Prepared {len(lc_docs)} LangChain documents.")

# 2. Create Vector Store (In-memory for this practice, or persisted)
# We use a unique collection name for practice
collection_name = "practice_rag_collection"

print("Creating Vector Store...")
vector_store = Chroma.from_documents(
    documents=lc_docs,
    embedding=embedding_function,
    collection_name=collection_name,
    # persist_directory="./practice_chroma_db" # Uncomment to save to disk
)

print("Vector Store created.")

# 3. Retrieval
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

query = "What are the common drug interactions?" # Change based on your PDF content
print(f"\nRetrieving for query: '{query}'")

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents.")
for i, doc in enumerate(retrieved_docs):
    print(f"\n[Doc {i+1}] Source: {doc.metadata['source']}")
    print(doc.page_content[:300] + "...") # Print first 300 chars



## 7. RAG Answer Function

Combine retrieval with Groq LLM to generate an answer.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Initialize LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

# Define Prompt
prompt_template = """
You are a helpful assistant. Use the following context to answer the question.
If you don't know the answer, just say that you don't know.

Context:
{context}

Question: {question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Define Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Execute
question = "What are the common drug interactions?" # Same query
print(f"Question: {question}\n")
print("Generating Answer...")

answer = rag_chain.invoke(question)

print("\n--- ANSWER ---")
print(answer)

